# Notebook 06: Dimensión de Unicidad

**Duración**: 30 minutos | **Nivel**: Principiante

## ¿Qué es Unicidad?

**Unicidad** asegura que no existan duplicados donde no deberían existir.

### Impacto:
- Doble facturación
- Métricas infladas
- Violación de integridad

In [ ]:
import great_expectations as gx
import pandas as pd

df = pd.read_csv("../data/ventas_sucias.csv")

print(f"Total registros: {len(df)}")
print(f"Order IDs únicos: {df['order_id'].nunique()}")
print(f"Duplicados: {len(df) - df['order_id'].nunique()}")

In [ ]:
# Configurar
context = gx.get_context(mode="ephemeral")
datasource = context.data_sources.add_pandas(name="ventas_ds")
asset = datasource.add_dataframe_asset(name="ventas")
batch_def = asset.add_batch_definition_whole_dataframe("batch_completo")

# Suite
suite = context.suites.add(gx.ExpectationSuite(name="unicidad"))

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeUnique(
        column="order_id",
        meta={"dimension": "Unicidad"}
    )
)

suite.save()

val_def = context.validation_definitions.add(
    gx.ValidationDefinition(data=batch_def, suite=suite, name="val_unicidad")
)

resultado = val_def.run(batch_parameters={"dataframe": df})
print(f"\n¿Validación exitosa?: {' SÍ' if resultado.success else ' NO'}")

## Detectar Duplicados

In [ ]:
# Encontrar duplicados
duplicados = df[df.duplicated(subset=['order_id'], keep=False)]
print(f"Registros duplicados: {len(duplicados)}")
print("\nEjemplos:")
print(duplicados.head(10))

##  Ejercicio

Valida que la combinación (customer_id, order_date) sea única.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: Crea una columna combinada o usa ExpectCompoundColumnsToBeUnique
pass

In [ ]:
context.build_data_docs()
context.open_data_docs()

##  Resumen

1.  `ExpectColumnValuesToBeUnique` detecta duplicados
2.  Identifica registros duplicados para investigación
3.  Valida unicidad en combinaciones de columnas

